In [1]:
import requests
import json
import uuid
from IPython.display import display_javascript, display_html, display
import pandas as pd

response = requests.get("http://api.openweathermap.org/data/2.5/forecast?id=5780993&APPID=e0c55e00bf021f6142de334823046e9e&units=imperial")
weather = response.content.decode("utf-8")
weatherDict = json.loads(weather)

In [2]:
print(json.dumps(weatherDict, indent=2))

{
  "message": 0.1691,
  "cnt": 40,
  "list": [
    {
      "dt_txt": "2016-11-13 00:00:00",
      "dt": 1478995200,
      "sys": {
        "pod": "n"
      },
      "wind": {
        "deg": 325.506,
        "speed": 6.55
      },
      "clouds": {
        "all": 88
      },
      "main": {
        "temp": 55.71,
        "temp_kf": 2.53,
        "pressure": 872.09,
        "sea_level": 1033.54,
        "grnd_level": 872.09,
        "temp_min": 51.15,
        "temp_max": 55.71,
        "humidity": 92
      },
      "weather": [
        {
          "id": 804,
          "main": "Clouds",
          "description": "overcast clouds",
          "icon": "04n"
        }
      ]
    },
    {
      "dt_txt": "2016-11-13 03:00:00",
      "dt": 1479006000,
      "sys": {
        "pod": "n"
      },
      "wind": {
        "deg": 354,
        "speed": 3.6
      },
      "clouds": {
        "all": 36
      },
      "main": {
        "temp": 49.21,
        "temp_kf": 1.9,
        "pressure": 872.52,
 

In [3]:
# Method for rendering collapsible JSON from: http://stackoverflow.com/questions/18873066/pretty-json-formatting-in-ipython-notebook

class RenderJSON(object):
    def __init__(self, json_data):
        if isinstance(json_data, dict):
            self.json_str = json.dumps(json_data)
        else:
            self.json_str = json
        self.uuid = str(uuid.uuid4())

    def _ipython_display_(self):
        display_html('<div id="{}" style="height: 600px; width:100%;"></div>'.format(self.uuid),
        raw=True)
        
        display_javascript("""
        require(["https://rawgit.com/caldwell/renderjson/master/renderjson.js"], function() {
        document.getElementById('%s').appendChild(renderjson(%s))
        });
        """ % (self.uuid, self.json_str), raw=True)
        
RenderJSON(weatherDict)

In [4]:
#pdWeather = pd.read_json(weatherDict)
#pdWeather

#pd.DataFrame(weatherDict["data"], columns=[x["label"] for x in weatherDict["fields"]])

timeList = []
maxTempList = []
minTempList = []
pressureList = []
humidityList = []
tempList = []
wind_speedList = []
windDegList = []
rainList = []

for weatherEntry in weatherDict["list"]:
    timeList.append(weatherEntry["dt_txt"])
    mainWeather = weatherEntry["main"]
    maxTempList.append(mainWeather["temp_max"])
    minTempList.append(mainWeather["temp_min"])
    pressureList.append(mainWeather["pressure"])
    humidityList.append(mainWeather["humidity"])
    tempList.append(mainWeather["temp"])
    windWeather = weatherEntry["wind"]
    wind_speedList.append(windWeather["speed"])
    windDegList.append(windWeather["deg"])

In [40]:
data = [('DateTime', timeList),
         ('MaxTemp', maxTempList),
         ('MinTemp', minTempList),
         ('Pressure', pressureList),
         ('Humidity', humidityList),
         ('Temperature', tempList),
         ('WindSpeed', wind_speedList), 
         ('WindDeg', windDegList)
         ]
weatherForecast = pd.DataFrame.from_items(data)
weatherForecast["DateTime"] = weatherForecast["DateTime"].apply(lambda x: str(x)[:10])

finalForecast = weatherForecast.groupby("DateTime").mean()

finalForecast["MinTemp"] = weatherForecast.groupby("DateTime").min()["MinTemp"]
finalForecast["MaxTemp"] = weatherForecast.groupby("DateTime").max()["MaxTemp"]
finalForecast["MaxPressure"] = weatherForecast.groupby("DateTime").max()["Pressure"]
finalForecast["MinPressure"] = weatherForecast.groupby("DateTime").min()["Pressure"]
finalForecast["MaxWindSpeed"] = weatherForecast.groupby("DateTime").max()["WindSpeed"]
finalForecast["MinWindSpeed"] = weatherForecast.groupby("DateTime").min()["WindSpeed"]

finalForecast

,MaxTemp,MinTemp,Pressure,Humidity,Temperature,WindSpeed,WindDeg,MaxPressure,MinPressure,MaxWindSpeed,MinWindSpeed
DateTime,,,,,,,,,,,
2016-11-13,55.71,40.36,873.47250,96.25,47.45250,4.13875,223.877000,875.13,872.09,6.55,2.04
2016-11-14,55.01,39.91,874.28625,95.25,45.40500,4.98375,179.003125,874.72,873.36,7.52,2.08
2016-11-15,55.38,40.69,870.86500,95.50,45.95500,2.64375,170.377475,873.37,865.83,3.38,1.72
2016-11-16,51.46,43.45,860.91250,95.00,45.86375,9.51375,192.751875,864.28,858.68,14.14,1.95
2016-11-17,39.12,33.52,862.66125,100.00,36.49500,8.48250,275.754750,866.12,859.59,10.11,4.47


In [41]:
finalForecast.info()

<class 'pandas.core.frame.DataFrame'>
Index: 5 entries, 2016-11-13 to 2016-11-17
Data columns (total 11 columns):
MaxTemp         5 non-null float64
MinTemp         5 non-null float64
Pressure        5 non-null float64
Humidity        5 non-null float64
Temperature     5 non-null float64
WindSpeed       5 non-null float64
WindDeg         5 non-null float64
MaxPressure     5 non-null float64
MinPressure     5 non-null float64
MaxWindSpeed    5 non-null float64
MinWindSpeed    5 non-null float64
dtypes: float64(11)
memory usage: 480.0+ bytes


In [43]:
finalForecast.describe()

,MaxTemp,MinTemp,Pressure,Humidity,Temperature,WindSpeed,WindDeg,MaxPressure,MinPressure,MaxWindSpeed,MinWindSpeed
count,5.000000,5.000000,5.00000,5.000000,5.000000,5.000000,5.000000,5.000000,5.000000,5.000000,5.000000
mean,51.336000,39.586000,868.43950,96.400000,44.234250,5.952500,208.352845,870.724000,865.910000,8.340000,2.452000
std,7.039711,3.662298,6.23383,2.066095,4.394315,2.926576,42.810458,5.126015,6.817562,4.038905,1.136693
min,39.120000,33.520000,860.91250,95.000000,36.495000,2.643750,170.377475,864.280000,858.680000,3.380000,1.720000
25%,51.460000,39.910000,862.66125,95.250000,45.405000,4.138750,179.003125,866.120000,859.590000,6.550000,1.950000
50%,55.010000,40.360000,870.86500,95.500000,45.863750,4.983750,192.751875,873.370000,865.830000,7.520000,2.040000
75%,55.380000,40.690000,873.47250,96.250000,45.955000,8.482500,223.877000,874.720000,872.090000,10.110000,2.080000
max,55.710000,43.450000,874.28625,100.000000,47.452500,9.513750,275.754750,875.130000,873.360000,14.140000,4.470000
